# 🏃 Cheerleading Pose Analysis — MediaPipe + Google Colab

> **🔒 Privacy Note:** Your videos stay in **your** Google Drive. They are mounted temporarily into this Colab runtime and processed locally. Nothing is uploaded to any third-party server. When you disconnect from Colab, every trace of your data is wiped from Google's VM. Only the output CSV is saved back to your Drive.

---

## How to use this notebook:

1. **Upload your videos** to your Google Drive, keeping the folder structure:
   ```
   MyDrive/cheer-ai/data/raw_videos/
   ├── standing/
   ├── high_v/
   ├── liberty/
   ├── t_jump/
   └── tuck_jump/
   ```
2. Run each cell in order (Ctrl+Enter or Shift+Enter)
3. The CSV will be saved to: `MyDrive/cheer-ai/data/processed_features/training_data_frames.csv`

## Step 1: Mount Google Drive & Install MediaPipe

In [ ]:
# Mount your Google Drive — follow the authorization prompt
from google.colab import drive
drive.mount('/content/drive')

# Install MediaPipe (Colab's Linux has full support)
!pip install -q mediapipe opencv-python numpy pandas

print("✅ Drive mounted and packages installed!")

## Step 2: Configure Paths

Adjust `DRIVE_DATA_DIR` if your folder structure differs.

In [ ]:
# === CONFIGURE THESE PATHS ===
DRIVE_DATA_DIR = '/content/drive/MyDrive/cheer-ai/data/raw_videos'
DRIVE_OUTPUT_CSV = '/content/drive/MyDrive/cheer-ai/data/processed_features/training_data_frames.csv'

import os
from pathlib import Path
out_dir = os.path.dirname(DRIVE_OUTPUT_CSV)
os.makedirs(out_dir, exist_ok=True)

print(f"Input dir:  {DRIVE_DATA_DIR}")
print(f"Output CSV: {DRIVE_OUTPUT_CSV}")

## Step 3: Imports & Landmark Indices

In [ ]:
import os, csv, json, math
from pathlib import Path
import cv2
import mediapipe as mp
import numpy as np

# MediaPipe Pose landmark indices (33 total, we define the 14 we need)
LEFT_SHOULDER  = 11
RIGHT_SHOULDER = 12
LEFT_ELBOW     = 13
RIGHT_ELBOW    = 14
LEFT_WRIST     = 15
RIGHT_WRIST    = 16
LEFT_HIP       = 23
RIGHT_HIP      = 24
LEFT_KNEE      = 25
RIGHT_KNEE     = 26
LEFT_ANKLE     = 27
RIGHT_ANKLE    = 28

VIDEO_EXTENSIONS = {'.mp4', '.mov', '.avi', '.webm', '.mkv'}

print("✅ Imports ready")

## Step 4: Vector Math Helpers

In [ ]:
def _vec(a, b):
    """3D vector from landmark a to b (X, Y, Z)."""
    return np.array([b.x - a.x, b.y - a.y, b.z - a.z])

def _norm(v):
    """Euclidean norm of a 3D vector."""
    return float(np.linalg.norm(v))

def _angle_between(v1, v2):
    """Angle in degrees between two 3D vectors (0-180)."""
    dot = float(np.dot(v1, v2))
    n1, n2 = _norm(v1), _norm(v2)
    if n1 < 1e-9 or n2 < 1e-9:
        return 0.0
    cos_theta = max(-1.0, min(1.0, dot / (n1 * n2)))
    return math.degrees(math.acos(cos_theta))

def _tilt_from_horizontal(vec):
    """Angle (0-90 deg) between vec and horizontal axis. 0 = level."""
    raw = _angle_between(vec, np.array([1.0, 0.0, 0.0]))
    return min(raw, 180.0 - raw)

print("✅ Math helpers defined")

## Step 5: 5 Biomechanical Parameters (all use X, Y, Z)

In [ ]:
def calc_shoulder_tilt(lm):
    """Shoulder Tilt (0-90 deg). 0 = level shoulders."""
    return _tilt_from_horizontal(_vec(lm[LEFT_SHOULDER], lm[RIGHT_SHOULDER]))

def calc_pelvic_tilt(lm):
    """Pelvic Tilt (0-90 deg). 0 = level hips."""
    return _tilt_from_horizontal(_vec(lm[LEFT_HIP], lm[RIGHT_HIP]))

def calc_trunk_shift(lm):
    """Trunk Shift (0-180 deg). 0 = vertical torso."""
    shoulder_mid = np.array([
        (lm[LEFT_SHOULDER].x + lm[RIGHT_SHOULDER].x) / 2,
        (lm[LEFT_SHOULDER].y + lm[RIGHT_SHOULDER].y) / 2,
        (lm[LEFT_SHOULDER].z + lm[RIGHT_SHOULDER].z) / 2,
    ])
    hip_mid = np.array([
        (lm[LEFT_HIP].x + lm[RIGHT_HIP].x) / 2,
        (lm[LEFT_HIP].y + lm[RIGHT_HIP].y) / 2,
        (lm[LEFT_HIP].z + lm[RIGHT_HIP].z) / 2,
    ])
    return _angle_between(shoulder_mid - hip_mid, np.array([0.0, 1.0, 0.0]))

def calc_knee_curvature(lm):
    """Knee Curvature (0-180 deg). 180 = straight legs. Average of L+R."""
    left_angle = _angle_between(
        _vec(lm[LEFT_HIP], lm[LEFT_KNEE]),
        _vec(lm[LEFT_KNEE], lm[LEFT_ANKLE]))
    right_angle = _angle_between(
        _vec(lm[RIGHT_HIP], lm[RIGHT_KNEE]),
        _vec(lm[RIGHT_KNEE], lm[RIGHT_ANKLE]))
    return (left_angle + right_angle) / 2.0

def calc_arm_misalignment(lm):
    """Arm Misalignment (0-90 deg). 0 = level arms. Avg of wrist + elbow."""
    wrist_angle = _tilt_from_horizontal(_vec(lm[LEFT_WRIST], lm[RIGHT_WRIST]))
    elbow_angle = _tilt_from_horizontal(_vec(lm[LEFT_ELBOW], lm[RIGHT_ELBOW]))
    return (wrist_angle + elbow_angle) / 2.0

print("✅ 5 parameter functions defined")

## Step 6: Utility Functions

In [ ]:
def severity_from_filename(filename):
    """Extract label from filename: _normal_ -> 'Normal', _bad_ -> 'Bad'."""
    basename = os.path.basename(filename)
    if '_normal_' in basename: return 'Normal'
    if '_bad_' in basename:    return 'Bad'
    return 'Unknown'

def landmarks_to_json(landmark_list):
    """Convert 33 landmarks to JSON string (x, y, z, 6 decimal places)."""
    data = []
    for lm in landmark_list:
        data.append({'x': round(lm.x, 6), 'y': round(lm.y, 6), 'z': round(lm.z, 6)})
    return json.dumps(data)

print("✅ Utility functions ready")

## Step 7: Process ALL Videos — Extract Every Frame's Pose

In [ ]:
# Discover all videos
video_files = []
for ext in VIDEO_EXTENSIONS:
    video_files.extend(Path(DRIVE_DATA_DIR).rglob(f'*{ext}'))
video_files = sorted(video_files)
print(f"Found {len(video_files)} videos.\n")

# Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=2,
    smooth_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

# Open CSV for writing (overwrites existing)
with open(DRIVE_OUTPUT_CSV, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['video_filename','move_name','severity_label','frame_number',
                     'shoulder_tilt','pelvic_tilt','trunk_shift','knee_curvature',
                     'arm_misalignment','landmarks_json'])
    
    frames_written, frames_skipped = 0, 0
    
    for vid_idx, vid_path in enumerate(video_files, 1):
        video_filename = vid_path.name
        move_name = vid_path.parent.name
        severity = severity_from_filename(video_filename)
        
        cap = cv2.VideoCapture(str(vid_path))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f"[{vid_idx}/{len(video_files)}] {move_name}/{video_filename} "
              f"({total_frames} frames) ... ", end='', flush=True)
        
        local_w, local_s, frame_num = 0, 0, 0
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            frame_num += 1
            
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(rgb)
            
            if results.pose_landmarks is None:
                local_s += 1
                continue
            
            lm = results.pose_landmarks.landmark
            
            try:
                st = calc_shoulder_tilt(lm)
                pt = calc_pelvic_tilt(lm)
                ts = calc_trunk_shift(lm)
                kc = calc_knee_curvature(lm)
                am = calc_arm_misalignment(lm)
                lm_json = landmarks_to_json(lm)
            except Exception:
                local_s += 1
                continue
            
            writer.writerow([video_filename, move_name, severity, frame_num,
                             round(st,4), round(pt,4), round(ts,4),
                             round(kc,4), round(am,4), lm_json])
            local_w += 1
        
        cap.release()
        frames_written += local_w
        frames_skipped += local_s
        print(f"{local_w} frames written" + (f", {local_s} skipped" if local_s else ""))

pose.close()

print(f"\n{'='*60}")
print(f"DONE! CSV saved to: {DRIVE_OUTPUT_CSV}")
print(f"Total frames written: {frames_written}")
print(f"Total frames skipped: {frames_skipped}")
print(f"{'='*60}")

## ✅ Done!

Your CSV is saved at the path shown above in your Google Drive. 

**🔒 Privacy Reminders:**
- Your videos never left your Google Drive
- The Colab runtime is ephemeral — disconnect to wipe all data from Google's servers
- The CSV contains only numbers (angles + landmark coordinates), no images or video data

**Next Steps:**
- Download the CSV from your Google Drive
- Run the notebook again anytime you add new videos — it will overwrite the CSV with fresh results
- When you're done, you can delete the notebook runtime via **Runtime → Disconnect and delete runtime**